In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmptm5eiho7".


In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#include <stdio.h>
#define N 4

// ***** Sección de código del device *****
__global__ void MatMultGPU(int* a, int* b, int* c)
{
	int k, sum = 0;
	int col = threadIdx.x + blockDim.x * blockIdx.x;
	int fil = threadIdx.y + blockDim.y * blockIdx.y;

	if (col < N && fil < N)
	{
		for (k = 0; k < N; k++)
		{
			sum += a[fil * N + k] * b[k * N + col];
		}
		c[fil * N + col] = sum;
	}
}

// ***** Sección de código del host *****

// ===== Código para mostrar los elementos de la matriz
void MatMostrar(char* Titulo, int a[N][N])
{
	printf("\n %s \n", Titulo);
	for (int i = 0; i < N; i++)
	{
		for (int j = 0; j < N; j++)
			printf("%d ", a[i][j]);
		printf("\n");
	}
}

int main()
{
	// -- Declarar variables
	int h_a[N][N], h_b[N][N], h_c[N][N];
	int* d_a, * d_b, * d_c;
	int cont;

	// -- Inicializar variables con datos
	for (int i = 0; i < N; i++)
	{
		cont = 0;
		for (int j = 0; j < N; j++)
		{
			h_a[i][j] = cont;
			h_b[i][j] = cont;
			cont++;
		}
	}

	int size = N * N * sizeof(int);

	cudaMalloc((void**)&d_a, size);
	cudaMalloc((void**)&d_b, size);
	cudaMalloc((void**)&d_c, size);

	cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
	cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

	dim3 dimGrid(1, 1);
	dim3 dimBlock(N, N);

	MatMultGPU << < dimGrid, dimBlock >> > (d_a, d_b, d_c);

	cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

	cudaFree(d_a);
	cudaFree(d_b);
	cudaFree(d_c);

	// imprimiendo
	MatMostrar("Matriz A", h_a);
	MatMostrar("Matriz B", h_b);
	MatMostrar("Matriz C", h_c);

	// -- Pausa antes de salir
	printf("\npulsa INTRO para finalizar...");
	fflush(stdin);
	char tecla = getchar();

	return 0;
        }



 Matriz A 
0 1 2 3 
0 1 2 3 
0 1 2 3 
0 1 2 3 

 Matriz B 
0 1 2 3 
0 1 2 3 
0 1 2 3 
0 1 2 3 

 Matriz C 
0 6 12 18 
0 6 12 18 
0 6 12 18 
0 6 12 18 

pulsa INTRO para finalizar...
